In [1]:
from pathlib import Path

FOLDER = Path("lab_csv")  # your folder in project root
CSV_FILES = list(FOLDER.glob("*.csv"))

if not CSV_FILES:
    raise FileNotFoundError(f"No .csv files found inside: {FOLDER.resolve()}")

BOM = b"\xef\xbb\xbf"  # UTF-8 BOM

fixed = []
skipped = []

for p in CSV_FILES:
    data = p.read_bytes()

    # If Excel-mojibake already exists, fix it safely by decoding/re-encoding.
    # We'll decode as UTF-8 first; if it fails, fallback to cp1252 (common Excel/Windows source).
    try:
        text = data.decode("utf-8")
    except UnicodeDecodeError:
        text = data.decode("cp1252", errors="replace")

    # Fix common mojibake cases (if any)
    text = text.replace("â‚¦", "₦")

    out = text.encode("utf-8")

    # Ensure BOM exists
    if not out.startswith(BOM):
        out = BOM + out

    p.write_bytes(out)
    fixed.append(p.name)

print("✅ Added UTF-8 BOM (and fixed any â‚¦ -> ₦) for:")
for name in fixed:
    print(" -", name)

✅ Added UTF-8 BOM (and fixed any â‚¦ -> ₦) for:
 - drugs_new.csv
 - general.csv
 - lab.csv
